
# MontePy Workshop

### Micah Gale<sup>1</sup>, Travis Labossiere-Hickman<sup>2</sup> 

Presented: April 18, 2026 at the ANS Student Conference 2026, College Station, TX

<sup>1</sup> Idaho National Laboratory; University of Wisconsin—Madison; Contact: [Micah.Gale@INL.gov](mailto:micah.gale@inl.gov)

<sup>2</sup> Idaho National Laboratory; University of Illinois, Urbana-Champaign; Contact: [Travis.LabossiereHickman@INL.gov](mailto:Travis.LabossiereHickman@INL.gov)

INL/CON-26-91533

# Table of Contents
* Introduction to MCNP
* Need for Automation
* Why MontePy?
* Introduction to MontePy documentation

# Introduction to MCNP
* Monte Carlo N-Particle (MCNP) developed by Los Alamos National Lab
* Code started in the 1960s<sup>1</sup>
* Very commonly taught in Monte Carlo classes
* Input syntax predates XML, JSON, YAML, SGML, oh my
* Syntax developed for punchcards
  
  <img alt="a stack of punch cards" src="https://upload.wikimedia.org/wikipedia/commons/2/26/Punched_card_program_deck.agr.jpg" width="300"></img>

  (By Arnold Reinhold CC BY-SA 3.0)

### MCNP input files are made of "inputs" not "cards" in MontePy documentation

<sup>1</sup> J. Kulesza et al., "MCNP Code Version 6.3.0 Theory & User Manual," LA-UR-22-30006, Rev. 1, September 2022.


```
Very important title
1 1 -19.81 -2 imp:n=1

2 so 10

m1 92235.80c 1.0
kcode 1000 1.5 50 200
ksrc 0 0 0
```


# Need for Automation
* We find these truths to be self-evident

## How many of you have done the following? 

### The lazy template

``` python
def update_radius(radius):
    with open("template.imcnp") as in_fh, open(f"new_{radius}.imcnp") as out_fh:
        for i, line in enumerate(in_fh):
            if i != 99:
                out_fh.write(line)
            else:
                out_fh.write(f"5 cz {radius}")
```

* `mcnp_pstudy.pl` would be a better tool here

### The Regular Expression Basher

``` python
import re

def change_mat_number(old_number, new_number):
     with open("old_victim.imcnp") as in_fh, open(f"new_victim.imcnp") as out_fh:
         for line in in_fh:
             if is_blue_moon_and_friday_13(line):
                 out_fh.write(re.sub(r"m(\d+)/////# how many backslaches???", f"m{new_number}"))
             elif is_dec_12_2012(line):
                 pass # so much more abuse
             else:
                 out_fh.write(line)
```

# Oh but, my script is different...
## Does it use context-free parsing in addition to regular expressions?

### If not: Then it is provably non-robust
* Regular expressions are *proven* to not be able to parse:
    * Parentheses groupings
    * Operator Precedence <sup>2</sup>

<sup>2</sup>K. D. Cooper and L. Torczon, Engineering a Compiler. San Francisco, CA: Morgan Kaufmann Publishers, 2004.


# Why MontePy
* Takes minutiae out of the automation
    * Does not require pre-templating
* Implements Context-Free parsing
     * Can preserve user formatting
     * Understands the order of operations for CSG
* Allows you to focus on semantics; not syntax!

In [ ]:
# note this install_montepy is only necessary for jupyterlite; You don't need to use it locally
from _config import install_montepy

await install_montepy()

import montepy

problem = montepy.read_input("models/pin_cell.imcnp")

densities = {1: 0.995, 3: 5.667, 5: 10.5}
for cell_num, importance in densities.items():
    problem.cells[cell_num].mass_density = importance
    print(problem.cells[cell_num].mcnp_str())

In [ ]:
# create a universe and fill another cell with it (hide)
universe = montepy.Universe(123)
problem.universes.append(universe)
# Slicing is inclusive, so this would include all cells with numbers between 1 and 5.
universe.claim(problem.cells[1:5])
print(problem.cells[1:5])

# Status of support


## Note: All Support Based on Public MCNP User Manual (6.2, 6.3.0, 6.3.1)
* We are aware MCNP has many undocumented "features," and will tolerate a lot of syntax abuse
    * MontePy will only ever support officially documented syntax

## Great fully Object-Oriented Support
   * Cells
   * Materials
   * Universes
   * Cylindrical, and Planar Surfaces (e.g., `CZ`, `PZ`)
      * Has `plane.location` and `cylinder.radius` properties


## Partial Object-Oriented Support
   * Other surfaces
        * Does not have meaningful properties for surface coefficients
        * A general `surf.surface_constants` can be used, as opposed to the specific `cylinder.radius`

## Limited Object-Oriented Support
   * Almost all Other Inputs
   * Can read information
   * Determined user can change input values if you know the location in the list
   * Has only properties for the classifier mnemonic


## Lacking any meaningful support
* Special complications
   * `DE`
     * It completely changes the syntax around `log`, will only be parroted.
  * `SDEF`
     * Has complicated syntax like: `SDEF PAR=FCEL=D3`.
  * `FMESH`
    * Uses commas like: `fmesh1:n vec=0,0,0`
   

## Unsupported Syntaxes
 * Vertical mode
 * Cell `Like n but...`
 * Will raise `UnsupportedFeatureException`

# Code Quality 
* Follows Modern Software Practices
    * Requires Peer reviews on Pull Requests
    * Uses Continuous Integration/Deployment (22 checks)
* Has extensive test suite
    * Over 950 tests
       * Tests are ran against 3 versions of pythons (3.11-3.14)
       * With two versions for both dependencies (numpy 2.0, 2.3 and sly 0.4, 0.5)
    * ~ 98% code coverage
* Stable versioning system.
   * Uses semantic versioning (`Major`, `Minor`, `Patch`). Only `Major` Releases should break old code.
   * Releases are immutable on [PyPI](https://pypi.org/project/montepy/) and [conda-forge](https://anaconda.org/conda-forge/montepy).
  * `pip install montepy`
  * `conda install conda-forge::montepy`

# MontePy Website
* It's on GitHub!
    * [github.com/idaholab/MontePy](https://github.com/idaholab/MontePy)
    * Go here for filing issues:
         * Bugs
         * Feature requests
         * Typos, etc
    * Star us!
    * Contribute on GitHub!




## Documentation is at [montepy.org](https://www.montepy.org/en/stable/)


In [ ]:
# note this %pip is only needed for running in jupyterlite online
%pip install ipython
from _config import IFrame

IFrame("https://www.montepy.org/en/stable/")

# Explain Top level menus

# Accessing Tutorials
* Use the getting started guide

In [ ]:
IFrame("https://www.montepy.org/en/stable/starting.html#specifying-nuclides")

# Accessing Doc Strings
## Structure
* Ordered by most commonly used groups of Objects
* Hopefully is intuitive
* Collections are special groups of `cells`, `materials`, etc.

# Finding Cell Doc-strings

In [ ]:
IFrame("https://www.montepy.org/en/stable/api/modules.html")

* Click cell
* Discuss methods versus properties
* Click on `geometry`
* Discuss the types

# Finding the Material Doc-strings

In [ ]:
IFrame("https://www.montepy.org/en/stable/api/modules.html")

* Click on material
* Discuss top level class and data
* Click `Material`
* Click on `add_nuclide`
* Discuss doc-string contract
* Discuss doc-string type hints
* Click on `append`
* Discuss `tuple[]` notation

# Note on Teaching Style
* We do not have time to make you experts
* Goal: teach self-efficacy
   * Introduce to MontePy organization dogma
   * Where to look for resources
   * How to approach software defined modeling

# Launching Jupyter Lite
* For this workshop we will use a version of Jupyter that runs entirely in your browser
* To launch it:
  1. Go to [montepy.org](https://montepy.org)
  2. Click [Launch jupyter in your browser](https://www.montepy.org/en/stable/lite/lab/)
  3. Start with notebook `1_fusion_radial_build.ipynb`
* Note: everything stored in the browser cache.
  * If you would like to save your progress download the files when you are done.

In [ ]:
IFrame("https://www.montepy.org/en/stable/")

# Acknowledgments 

Work supported through the Advanced Fuels Campaign (AFC) under DOE Idaho Operations Office Contract DE-AC07-05ID14517. The authors wish to thank the U.S. Department of Energy Office of Isotope R&D and Production for their vital and continued support and funding of the Co-60 program at INL under Contract No. DE-AC07-05ID14517. Co-60 is sold by the National Isotope Development Center (NIDC). Quotes on Co-60 can be obtained from NIDC at www.isotopes.gov/products/cobalt. This research made use of Idaho National Laboratory's High Performance Computing systems located at the Collaborative Computing Center and supported by the Office of Nuclear Energy of the U.S. Department of Energy and the Nuclear Science User Facilities under Contract No. DE-AC07-05ID14517. Work supported through the INL Laboratory Directed Research & Development (LDRD) Program
under DOE Idaho Operations Office Contract DE-AC07-05ID14517.

# Questions ?

## Contact Us
* Documentation: [montepy.org](https://www.montepy.org/en/stable/)
* Repository: [github.com/idaholab/MontePy](https://github.com/idaholab/MontePy)
* Lead Developer: [Micah.Gale@INL.gov](mailto:micah.gale@inl.gov)
  
![QR code for montepy.org](figs/montepy_qr.png)